# 📈 Algoritmo de Trading Automatizado con Acciones Argentinas

Este notebook realiza:
- Backtest técnico con RSI, MACD, EMAs
- Simulación de señales de compra
- Opcional: alertas por Telegram

In [ ]:
!pip install yfinance ta

In [ ]:
# 📌 Importación de librerías
import yfinance as yf
import pandas as pd
import ta
import matplotlib.pyplot as plt
import requests

In [ ]:
# 📌 Configuración inicial
capital = 677414
porcentaje_inversion = 0.15
acciones = ['GGAL.BA', 'YPFD.BA', 'TXAR.BA', 'BMA.BA', 'VALO.BA', 'MIRG.BA']

# 🔔 Telegram (opcional)
telegram_token = "AQUÍ_TU_TOKEN"
chat_id = "AQUÍ_TU_CHAT_ID"

def enviar_alerta_telegram(mensaje):
    if telegram_token == "AQUÍ_TU_TOKEN": return  # No configurado
    url = f"https://api.telegram.org/bot{telegram_token}/sendMessage"
    payload = {'chat_id': chat_id, 'text': mensaje}
    r = requests.post(url, data=payload)
    return r.status_code == 200

In [ ]:
# 📌 Backtest por ticker con RSI, MACD, EMA

def backtest_estrategia(ticker):
    df = yf.download(ticker, period='6mo', interval='1d')
    df.dropna(inplace=True)

    df['rsi'] = ta.momentum.RSIIndicator(df['Close']).rsi()
    macd = ta.trend.MACD(df['Close'])
    df['macd_diff'] = macd.macd_diff()
    df['ema20'] = ta.trend.EMAIndicator(df['Close'], 20).ema_indicator()
    df['ema50'] = ta.trend.EMAIndicator(df['Close'], 50).ema_indicator()

    df['signal'] = (df['rsi'] < 40) & (df['macd_diff'] > 0) & (df['ema20'] > df['ema50'])
    df['comprar'] = df['signal'] & (~df['signal'].shift(1).fillna(False))
    compras = df[df['comprar']]

    if compras.empty:
        print(f"❌ {ticker} no tuvo señales en los últimos 6 meses.")
        return None

    print(f"✅ {ticker}: {len(compras)} señales de compra detectadas.")
    display(compras[['Close', 'rsi', 'macd_diff', 'ema20', 'ema50']].tail())

    plt.figure(figsize=(14, 6))
    plt.plot(df.index, df['Close'], label='Precio')
    plt.scatter(compras.index, compras['Close'], label='Señal de Compra', color='green', marker='^', s=100)
    plt.title(f"{ticker} - Señales de Compra")
    plt.legend()
    plt.grid()
    plt.show()

    # Enviar alerta por Telegram
    if not compras.empty:
        ultimo = compras.iloc[-1]
        mensaje = f"📈 Señal de compra: {ticker}\nPrecio: ${ultimo['Close']:.2f}"
        enviar_alerta_telegram(mensaje)

    return compras

In [ ]:
# 📊 Ejecutar backtest
for ticker in acciones:
    backtest_estrategia(ticker)